# YouTube tool-calling agent — walkthrough

Tools → the manual loop → a fixed LCEL chain → a recursive chain → `create_agent` → the exercises' report. Runs **offline by default**: a rule-based stand-in model and a bundled catalogue of made-up videos (no API key, no network, no cost). Set `PROVIDER = "gemini"` / `"openai"` and `SOURCE = "live"` for the real thing.

In [1]:
PROVIDER, SOURCE = "offline", "sample"   # "gemini" | "openai" | "offline";  "live" | "sample"

from dotenv import load_dotenv
load_dotenv()
from langchain_core.messages import HumanMessage
from yt_agent import (TOOLS, extract_video_id, fetch_transcript, get_full_metadata, get_llm, set_source,
                      execute_tool, run_tool_loop, build_fixed_chain, build_recursive_chain, build_agent, ask,
                      collect_tool_calls, collect_video_data, build_report_prompt)
from yt_agent.cli import print_step, print_trace
set_source(SOURCE)
llm = get_llm(PROVIDER)
URL = "https://www.youtube.com/watch?v=SmplAI00001"

## 1. The tools (what the model sees)

In [2]:
for t in TOOLS:
    print(f"{t.name:20} {list(t.args)}")

extract_video_id     ['url']
fetch_transcript     ['video_id', 'language', 'max_chars']
search_youtube       ['query', 'max_results']
get_full_metadata    ['url']
get_trending_videos  ['region_code', 'max_results']
get_thumbnails       ['url']


In [3]:
vid = extract_video_id.invoke({"url": URL})
print(vid)
print(fetch_transcript.invoke({"video_id": vid, "max_chars": 120}))
print(get_full_metadata.invoke({"url": "https://youtu.be/NotThere123"}))   # errors are data

SmplAI00001
{'video_id': 'SmplAI00001', 'language': 'en', 'transcript': 'A language model only produces text, so on its own it cannot check a database or do exact math. Tool calling fixes that ', 'truncated': True}
{'error': "Video 'NotThere123' not found in the sample catalogue."}


## 2. The manual loop: the model asks, your code runs the tool, the result goes back with the call id

In [4]:
result = run_tool_loop(llm, f"I want to summarize youtube video: {URL} in english", on_step=print_step)

Human        I want to summarize youtube video: https://www.youtube.com/watch?v=SmplAI00001 in english
AI → tool    extract_video_id({'url': 'https://www.youtube.com/watch?v=SmplAI00001'})
Tool         extract_video_id → SmplAI00001
AI → tool    fetch_transcript({'video_id': 'SmplAI00001', 'language': 'en'})
Tool         fetch_transcript → {"video_id": "SmplAI00001", "language": "en", "transcript": "A language model only produces text, so on its own it cannot check a database or do exact math. Tool calling fixes that … (+389 chars)
AI answer    Summary (extractive, offline model):
             - A language model only produces text, so on its own it cannot check a database or do exact math.
             - Tool calling fixes that by letting the model ask for a function by name.
             - Your code runs the function and sends the result back with the same call id.


## 3. Fixed chain (two tool rounds, wired with `RunnablePassthrough.assign`)
Fine for *ID → transcript → summary*; stuck when a task needs a third round.

In [5]:
fixed = build_fixed_chain(llm)
for q in [f"Summarize {URL}", "Show top 3 US trending videos with metadata and thumbnails"]:
    out = fixed.invoke({"query": q})
    print(f"complete={out['complete']!s:5}  answer={out['answer'][:70]!r}")

complete=True   answer='Summary (extractive, offline model):\n- A language model only produces '
complete=False  answer=''


## 4. Recursive chain: keep going until the model stops calling tools

In [6]:
messages = build_recursive_chain(llm).invoke({"query": "Show top 3 US trending videos with metadata and thumbnails"})
print_trace(messages)

Human        Show top 3 US trending videos with metadata and thumbnails
AI → tool    get_trending_videos({'region_code': 'US'})
Tool         get_trending_videos → [{"title": "Tool Calling in 10 Minutes", "video_id": "SmplAI00001", "url": "https://www.youtube.com/watch?v=SmplAI00001", "channel": "Sample Academy", "views": 184200, "duration":  … (+556 chars)
AI → 3 tools get_full_metadata({'url': 'https://www.youtube.com/watch?v=SmplAI00001'})
AI → 3 tools get_full_metadata({'url': 'https://www.youtube.com/watch?v=SmplFT00004'})
AI → 3 tools get_full_metadata({'url': 'https://www.youtube.com/watch?v=SmplCK00003'})
Tool         get_full_metadata → {"video_id": "SmplAI00001", "title": "Tool Calling in 10 Minutes", "channel": "Sample Academy", "views": 184200, "duration": 612, "likes": 9100, "comments": 412, "published": "2026 … (+206 chars)
Tool         get_full_metadata → {"video_id": "SmplFT00004", "title": "20-Minute Morning Mobility", "channel": "Move Daily", "views": 655000, "duration

## 5. The same loop, prebuilt: `create_agent`

In [7]:
print_trace(ask(build_agent(llm), "Search for videos about python testing and summarize the top result")["messages"])

Human        Search for videos about python testing and summarize the top result
AI → tool    search_youtube({'query': 'python testing', 'max_results': 3})
Tool         search_youtube → [{"title": "Python Testing with pytest", "video_id": "SmplPY00006", "url": "https://www.youtube.com/watch?v=SmplPY00006", "channel": "Sample Academy", "views": 271300, "duration":  … (+191 chars)
AI → tool    fetch_transcript({'video_id': 'SmplPY00006'})
Tool         fetch_transcript → {"video_id": "SmplPY00006", "language": "en", "transcript": "A pytest test is just a function whose name starts with test. Plain assert statements give clear failure messages witho … (+268 chars)
AI answer    Top result: "Python Testing with pytest" (Sample Academy). Summary (extractive, offline model):
             - A pytest test is just a function whose name starts with test.
             - Plain assert statements give clear failure messages without extra helpers.
             - Fixtures provide shared setup such as te

## 6. Exercises: collect everything, then one LLM call (no tool calling)

In [8]:
data = collect_video_data("https://www.youtube.com/watch?v=SmplCK00003")
print(data["video_id"], "|", data["metadata"]["title"], "|", len(data["transcript"]), "chars |", len(data["thumbnails"]), "thumbnails")
response = llm.invoke([HumanMessage(content=build_report_prompt(data))])
print("\n===== VIDEO ANALYSIS =====\n")
print(response.content)

SmplCK00003 | Perfect Sourdough at Home | 363 chars | 4 thumbnails

===== VIDEO ANALYSIS =====

Analysis of "Perfect Sourdough at Home" by Kitchen Lab (offline demo model, templated):

1. Summary:
   - Sourdough needs only flour, water, salt and an active starter.
   - Feed the starter the night before so it is bubbly when you mix the dough.
   - Stretch and fold the dough every half hour to build strength without kneading.
   - A long cold proof in the fridge gives better flavour and an easier bake.
2. Main topics: perfect, sourdough, starter, dough
3. Audience: viewers interested in perfect sourdough at home; beginner-friendly tone.
4. Performance: 2410000 views with a 3.7% like-to-view ratio; length 22 min 0 s.


## 7. Evaluate: check the tool calls, not the wording

In [9]:
checks = [
    (f"Summarize {URL}", ["extract_video_id", "fetch_transcript"]),
    ("Get top 3 youtube videos in India and their metadata", ["get_trending_videos"] + ["get_full_metadata"] * 3),
]
passed = 0
for q, expected in checks:
    names = [c["name"] for c in run_tool_loop(llm, q).tool_calls]
    ok = names == expected
    passed += ok
    print("✅" if ok else "❌", q[:55], "->", names)
print(f"\n{passed}/{len(checks)} checks passed")

✅ Summarize https://www.youtube.com/watch?v=SmplAI00001 -> ['extract_video_id', 'fetch_transcript']
✅ Get top 3 youtube videos in India and their metadata -> ['get_trending_videos', 'get_full_metadata', 'get_full_metadata', 'get_full_metadata']

2/2 checks passed
